# Machine-Learning-Assisted History Matching and Production Forecasting
### Team SparkLab — ExxonMobil DataWorks Challenge 2026

This notebook is the technical record of our submission. It follows the workflow of the challenge: *learn a fast model of the reservoir simulator → match the observed field history → forecast production to 01/01/2028*. Every number and figure below is produced by the code in this notebook and in `src/`.

## Summary

| | |
|---|---|
| **Approach** | A proxy model (PCA + Gaussian process) replaces the simulator; it is used to history-match the observed field data and to generate 10 forecast cases |
| **Proxy accuracy** | On unseen simulation cases the mean relative error is **0.12 %** for gas and oil cumulative, **0.37 – 0.56 %** for water cumulative (validation / test) |
| **History match** | Best misfit **0.0160**, lower than every one of the 100 simulated cases (best: 0.0171) |
| **Forecast** | 10 distinct near-optimal parameter sets, quarterly cumulative gas / oil / water from 04/01/2008 to **01/01/2028** (80 quarters per case) |
| **Deliverable** | `09 Template Deliverable` filled for all 10 cases, including Case 1 through 01/01/2028 |

**Contents:** 1 Setup · 2 Data · 3 Feature engineering · 4 Proxy model · 5 Training · 6 Validation · 7 Final model · 8 History matching · 9 Selected cases · 10 Forecast · 11 Template · 12 Alternative models · 13 Assumptions and limitations · 14 Reproducibility

## 1. Setup
Libraries, project paths and the scripts in `src/` that implement each step.

In [ ]:
# SECTION 1 - SETUP: import libraries, set project paths, load the scripts in src/ (proxy, hm, decline)
import sys, os, pickle, warnings, inspect
warnings.filterwarnings("ignore")
# project root (works from the notebooks/ folder or the project folder)
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd())=="notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import openpyxl
from paths import DATA, MODELS, OUT
from proxy import Proxy, X, Yall, SER, split, feat
import hm
from hm import misfit
from decline import forecast
plt.rcParams.update({"figure.dpi":110,"axes.grid":True,"grid.alpha":.3})
print("project root:", ROOT)

## 2. Data
`data.npz` is built from the challenge files by `src/prep.py`. Each simulation case is one row:

| rows | content |
|---|---|
| 0 – 69 | **training** cases (Case 1–70) |
| 70 – 84 | **validation** cases (Case 71–85) |
| 85 – 99 | **test** cases (Case 86–100) |
| 100 | the **observed field history** (`RAMP_History`) — the target of the history match |

* **Inputs (4 per case):** fault transmissibility, porosity multiplier, permeability multiplier, aquifer pore volume.
* **Outputs (6 curves per case):** gas, oil and water **cumulative** production and gas, oil and water **rate**, each on 41 quarterly nodes from 1998 to 2008 (6 × 41 = 246 values).
* The production sheets contain irregular daily records (about 11,000 rows, roughly 170 values per case); `prep.py` linearly interpolates each case onto the common quarterly grid.

In [ ]:
# SECTION 2 - DATA: load the 100 cases + observed history; summarise the 4 input settings (min / mean / max)
names = ["fault","poro_mult","perm_mult","aquifer_pv"]
print("X:", X.shape, " Yall:", Yall.shape)
print(pd.DataFrame(X[:100], columns=names).describe().round(3).loc[["min","mean","max"]])
t = 1998 + np.arange(41)/4

In [ ]:
# SECTION 2 - DATA: plot all 100 simulated cases (grey) against the observed history (red)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for a, (i, k, lab) in zip(ax, [(0,"gas_cum","Gas cum [MSCF]"),(1,"oil_cum","Oil cum [STB]"),(2,"water_cum","Water cum [STB]")]):
    Y = split(Yall)[k]
    for r in Y[:100]: a.plot(t, r, color="grey", alpha=.25, lw=.8)
    a.plot(t, Y[100], color="crimson", lw=2.2, label="observed history")
    a.set_title(lab); a.set_xlabel("year")
ax[0].legend(); plt.suptitle("100 simulated cases (grey) vs observed history (red)", y=1.03); plt.show()

## 3. Feature engineering
* `fault` and `poro_mult` are used directly. `perm_mult` and `aquifer_pv` are **log-transformed**, because they act multiplicatively.
* All inputs and all 246 outputs are **standardised** (zero mean, unit variance).
* The 3-D grid files were extracted for all 100 cases and analysed (`outputs/grid_summary.csv`). Beyond the four settings they contain no additional case-specific information, so the proxy is driven by the four settings alone.

In [ ]:
# SECTION 3 - FEATURE ENGINEERING: print feat(), the log transform applied to permeability and aquifer volume
print(inspect.getsource(feat))

## 4. Proxy model: PCA + Gaussian process
Only 70 training simulations are available, so we use a small-data method that interpolates smoothly and is fast to evaluate.

```
4 settings -> standardise -> GP_1 .. GP_10 -> 10 PCA scores -> inverse PCA -> 246 curve values
```

* **PCA** (10 components) compresses the 246 strongly correlated outputs and removes noise; the 10 components retain 99.95 % of the variance.
* **One Gaussian process per component**, kernel `Constant × RBF + White noise`, with a separate length scale per input (automatic relevance determination), 4 optimiser restarts, `normalize_y=True`.
* The proxy evaluates in milliseconds, which makes a 100,000-sample history-matching search practical.

In [ ]:
# SECTION 4 - PROXY MODEL: print the Proxy class (standardise -> PCA -> one Gaussian process per component)
import inspect; print(inspect.getsource(Proxy))

## 5. Training
The proxy is trained on Case 1–70 only; Case 71–100 are held out.

In [ ]:
# SECTION 5 - TRAINING: fit the proxy on cases 1-70 only (cases 71-100 stay unseen)
tr, va, te = slice(0,70), slice(70,85), slice(85,100)
# 10 PCA components; one Gaussian process per component
proxy_tr = Proxy(10).fit(X[tr], Yall[tr])
print("trained: 10 PCA components, explained variance =", proxy_tr.pca.explained_variance_ratio_.sum().round(5))
print("PCA explained variance by component:", proxy_tr.pca.explained_variance_ratio_.round(4))

## 6. Validation and test
The held-out cases are used to measure how well the proxy reproduces the simulator. **Error metric:** for each curve, the sum of absolute errors divided by the sum of absolute true values, averaged over cases (%).

In [ ]:
# SECTION 6 - VALIDATION: relative error (%) on validation (71-85) and test (86-100) cases, per series
def rel_err(P, T):
    P, T = split(P), split(T)
    return {k: float(np.mean(np.abs(P[k]-T[k]).sum(1)/np.abs(T[k]).sum(1))*100) for k in SER}
# collect validation and test errors in one table
rows = {"validation (71–85)": rel_err(proxy_tr.predict(X[va]), Yall[va]),
        "test (86–100)":      rel_err(proxy_tr.predict(X[te]), Yall[te])}
err = pd.DataFrame(rows).T.round(2); err.columns = [c.replace("_"," ") for c in err.columns]
err.style.format("{:.2f}%") if False else display(err.astype(str)+"%")

In [ ]:
# SECTION 6 - VALIDATION: predicted vs simulated plots (final cumulative, and one example case)
Pv = split(proxy_tr.predict(X[va])); Tv = split(Yall[va])
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for a, k, lab in zip(ax, ["gas_cum","oil_cum","water_cum"], ["Gas cum","Oil cum","Water cum"]):
    a.scatter(Tv[k][:, -1], Pv[k][:, -1], s=28)
    lim = [min(Tv[k][:, -1].min(), Pv[k][:, -1].min()), max(Tv[k][:, -1].max(), Pv[k][:, -1].max())]
    a.plot(lim, lim, "k--", lw=1); a.set_title(f"{lab} at 2008: predicted vs simulated"); a.set_xlabel("simulator"); a.set_ylabel("proxy")
plt.show()
j = 3  # one validation case
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
for a, k in zip(ax, ["gas_cum","oil_cum","water_cum"]):
    a.plot(t, Tv[k][j], "o-", ms=3, label="simulator"); a.plot(t, Pv[k][j], "r-", label="proxy"); a.set_title(f"{k} — validation case {71+j}")
ax[0].legend(); plt.show()

**Result.** Gas and oil cumulative are reproduced to about **0.12 %**, water cumulative to **0.4 – 0.6 %**, and the rates to **0.3 – 0.8 %**. Water is the hardest series because water breakthrough is more non-linear. These errors compare the proxy with the *simulator* on unseen cases; they are not errors against real field data.

## 7. Final model
After validation the proxy is refitted on **all 100 simulation cases** (`src/hm.py`, saved to `models/proxy_final.pkl`) and used for history matching.

In [ ]:
# SECTION 7 - FINAL MODEL: load the proxy refitted on all 100 cases (made by src/hm.py)
proxy = pickle.load(open(MODELS/"proxy_final.pkl","rb"))
print(type(proxy).__name__, "components:", proxy.nc)

## 8. History matching
**Objective:** find the four settings whose predicted curves best reproduce the observed history.

**Misfit function** — weighted root-mean-square of normalised differences over the six series:
* cumulative series (gas, oil, water) have weight 1; rate series have weight 0.5;
* each series is normalised by its maximum over the 100 simulations, so all series contribute on a comparable scale;
* the first 6 nodes (1998 – 1999 start-up) of the rate series are excluded, because a one-quarter timing offset in the start-up would otherwise dominate the score.

**Search:** 100,000 random parameter sets drawn uniformly inside the training bounds are scored by the proxy; the best 20 are refined with the derivative-free Nelder–Mead optimiser.

In [ ]:
# SECTION 8 - HISTORY MATCHING: print the misfit function (weighted RMS vs observed history)
import inspect; print(inspect.getsource(hm.misfit))

In [ ]:
# SECTION 8 - HISTORY MATCHING: results of the 100,000-sample search and the Nelder-Mead refinement
# S: sampled settings, mf: their misfit, mfc: misfit of the 100 simulated cases, best: refined sets
S, mf, mfc, lo, hi, best = pickle.load(open(OUT/"hm_samples.pkl","rb"))
print("random samples scored:", len(S))
print("best simulated case misfit  :", mfc[:100].min().round(4), "(case", int(np.argmin(mfc[:100]))+1, ")")
print("median simulated case misfit:", np.median(mfc[:100]).round(4))
print("best proxy sample           :", mf.min().round(4))
print("best after Nelder–Mead      :", round(best[0][0],4))
print("best parameters             :", dict(zip(names, np.round(best[0][1],4))))

In [ ]:
# SECTION 8 - HISTORY MATCHING: misfit against each setting (shows non-uniqueness)
fig, ax = plt.subplots(1, 4, figsize=(15, 3.2))
for k, a in enumerate(ax):
    a.scatter(S[::20, k], mf[::20], s=2, alpha=.4); a.set_xlabel(names[k]); a.set_ylabel("misfit" if k==0 else "")
    if k>=2: a.set_xscale("log")
plt.suptitle("Misfit vs each setting (20 000 of the 100 000 samples) — many different combinations fit similarly: non-uniqueness", y=1.04); plt.show()

## 9. Ten distinct near-optimal parameter sets
History matching is **non-unique**: the scatter plot above shows that quite different parameter combinations fit the history equally well. Rather than ten copies of one answer, we select a *diverse* set — the refined best set plus the next-best samples that are at least 0.2 apart (normalised distance in fault / permeability / aquifer space) from every set already chosen. All ten have essentially the same misfit (≈ 0.016).

In [ ]:
# SECTION 9 - SELECTED CASES: the 10 distinct near-optimal parameter sets and their misfit
sel = np.load(OUT/"selected.npz")
# one row per selected case: its 4 settings and its misfit
tbl = pd.DataFrame(sel["theta"], columns=names); tbl.insert(0, "case", range(1,11)); tbl["misfit"] = sel["mis"]
display(tbl.round(4))

In [ ]:
# SECTION 9 - SELECTED CASES: the 10 matched models (blue) against the observed history (red)
H = split(Yall)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
Pp = split(sel["hist_pred"])
for a, k in zip(ax, ["gas_cum","oil_cum","water_cum"]):
    for r in Pp[k]: a.plot(t, r, color="tab:blue", alpha=.5, lw=1)
    a.plot(t, H[k][100], "r-", lw=2.2, label="observed"); a.set_title(k)
ax[0].plot([], [], color="tab:blue", label="10 matched models"); ax[0].legend(); plt.show()

## 10. Forecast to 01/01/2028
The proxy is trained on 1998 – 2008, so it is not used beyond 2008. From **01/01/2008 (node 40)** each matched model's own curves are extrapolated with standard reservoir-engineering decline methods, fitted on the **last 12 quarters**:

* **Oil rate:** average of an **exponential** decline `q = q₀·e^(−Dt)` and an **Arps hyperbolic** decline `q = q₀ / (1 + bDt)^(1/b)` with `b` limited to 0.01 – 1.5.
* **Gas rate:** gas–oil ratio (GOR) trended linearly, `q_g = GOR(t) · q_o`.
* **Water rate:** `ln(WOR)` linear in cumulative oil, `q_w = WOR · q_o`.
* **Cumulative production:** trapezoidal integration of the rates, starting from the matched model's 2008 cumulative.

The variants (window length, exponential vs hyperbolic, WOR basis) were compared by back-testing on the 100 simulated cases (`python src/decline.py`).

In [ ]:
# SECTION 10 - FORECAST: print the decline-curve function used after 2008 (oil, gas, water)
import inspect; print(inspect.getsource(forecast))

In [ ]:
# SECTION 10 - FORECAST: plot observed history + the 10 forecasts to 01/01/2028
# res: 10 cases x [oil, gas, water] x 80 quarters (2008-04-01 ... 2028-01-01)
res = np.load(OUT/"fc20.npz")["res"]          # 10 cases x [oil, gas, water] x 80 quarters
tf = 2008 + (np.arange(1, 81))/4
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (i, lab, k) in zip(ax, [(1,"Gas production cumulative [MSCF]","gas_cum"),(0,"Oil production cumulative [STB]","oil_cum"),(2,"Water production cumulative [STB]","water_cum")]):
    a.plot(t, H[k][100], "k-", lw=2, label="observed history")
    for j in range(10): a.plot(np.r_[t[-1], tf], np.r_[Pp[k][j][-1], res[j, i]], lw=1, alpha=.8)
    a.axvline(2008, color="grey", ls=":"); a.set_title(lab, fontsize=10); a.set_xlabel("year")
ax[0].legend(); plt.suptitle("SparkLab forecast: 10 cases to 01/01/2028", y=1.02); plt.show()

In [ ]:
# SECTION 10 - FORECAST: cumulative gas / oil / water at 2028 for each case, and the spread between cases
# values at 01/01/2028, the last row of the template
final = pd.DataFrame({"gas cum [MSCF]": res[:,1,-1], "oil cum [STB]": res[:,0,-1], "water cum [STB]": res[:,2,-1]}, index=[f"Case {i}" for i in range(1,11)])
display(final.round(0).astype(int))
print("spread across the 10 cases (max/min -1):", ((final.max()/final.min()-1)*100).round(2).to_dict(), "%")

**Physical consistency check** — every cumulative curve must be non-negative and non-decreasing:

In [ ]:
# SECTION 10 - FORECAST CHECK: every cumulative curve must be non-negative and non-decreasing
assert (res >= 0).all() and (np.diff(res, axis=2) >= -1e-6).all()
print("all 30 curves are monotone non-decreasing: OK")

## 11. Deliverable: filled template
`09 Template Deliverable.xlsx` has one sheet per case (`Case 1` … `Case 10`) with *Date | Gas cumulative [MSCF] | Oil cumulative [STB] | Water cumulative [STB]* for 80 quarterly dates ending 01/01/2028. `src/forecast_run.py` writes **only the numeric cells**; dates, headers and number formats of the template are left untouched, and the script asserts that every row's date matches the expected quarter.

In [ ]:
# SECTION 11 - TEMPLATE: write the forecast into the official template (numeric cells only) and re-read it to check
# open the unchanged template; only the numeric cells are overwritten
wb = openpyxl.load_workbook(DATA/"09 Template Deliverable.xlsx")
exp = [(y if mo!=1 else y+1, mo) for y in range(2008, 2028) for mo in (4,7,10,1)]
for j in range(10):
    ws = wb[f"Case {j+1}"]
    assert ws.cell(1,2).value.startswith("Gas") and ws.cell(1,3).value.startswith("Oil") and ws.cell(1,4).value.startswith("Water")
    for r in range(80):
        v = ws.cell(r+2, 1).value
        got = (int(v.split("/")[2]), int(v.split("/")[0])) if isinstance(v, str) else (v.year, v.month)
        assert got == exp[r]
        ws.cell(r+2,2).value = float(res[j,1,r]); ws.cell(r+2,3).value = float(res[j,0,r]); ws.cell(r+2,4).value = float(res[j,2,r])
os.makedirs(OUT, exist_ok=True)
import tempfile; _tmp=os.path.join(tempfile.gettempdir(),"notebook_check_template.xlsx"); wb.save(_tmp)
chk = openpyxl.load_workbook(_tmp)["Case 1"]
print([c.value for c in chk[1][:4]]); print([c.value for c in chk[81][:4]])

## 12. Alternative models
Using the same split, targets and error metric (`src/compare_models.py`), we compared the Gaussian-process proxy with polynomial ridge regression, a tuned neural network (MLP), random forest and gradient boosting. Table: validation-set mean relative error (%).

In [ ]:
# SECTION 12 - ALTERNATIVE MODELS: validation error (%) of the Gaussian process vs other ML models
import json
# results written by src/compare_models.py
cm = json.load(open(OUT/"model_comparison.json"))
rows = []
for m_ in cm: rows.append({"model": m_["model"], **{f"val {k} (%)": round(v,2) for k,v in m_["val"].items() if k in ("gas_cum","oil_cum","water_cum")}})
display(pd.DataFrame(rows))

The Gaussian process is the most accurate model on every cumulative series; the tuned neural network is the closest alternative (about 0.18 % against 0.12 % for oil cumulative), while tree ensembles and polynomial ridge are several times less accurate. With only 70 training simulations, larger deep networks were not pursued.

## 13. Assumptions and limitations
* **Extrapolation beyond 2008.** The proxy is trained on 1998 – 2008; the 2008 – 2028 forecast is a decline-curve extrapolation. Back-tests cover only 1 – 3 years ahead, so confidence decreases with horizon.
* **Start of the forecast.** The forecast starts from the matched model's predicted rates at 01/01/2008. At that date these rates differ from the observed field rates by about +13.7 % (oil), +13.8 % (gas) and −9.8 % (water); the match is strongest on the cumulative curves.
* **Spread of the 10 cases.** The ten forecasts agree within about 1 – 2 % at 2028 because they share one extrapolation method, so this spread reflects parameter non-uniqueness and under-represents total forecast uncertainty.
* **Selection of cases.** "Best 10 forecast cases" is interpreted as ten optimised, mutually distinct parameter sets (rather than ten of the 100 supplied cases).
* **Data use.** The challenge data are used only for this challenge and are not included in any public repository.

## 14. Reproducibility
All steps are scripts in `src/` and run in this order (paths are set in `src/paths.py`):
```
python src/00_extract.py <folder>   # read the challenge Excel files  (once)
python src/prep.py                  # interpolate to quarterly grid -> data/data.npz
python src/proxy.py                 # proxy validation for 6 / 8 / 10 / 12 PCA components
python src/hm.py                    # refit proxy on 100 cases, history match
python src/forecast_run.py          # select 10 sets, forecast to 2028, fill template
python src/compare_models.py        # alternative models
python src/build_powerbi.py         # data for the Power BI dashboard
python src/make_plot.py             # overview figure
```
Environment: Python 3, `numpy`, `pandas`, `scipy`, `scikit-learn`, `matplotlib`, `openpyxl` (see `requirements.txt`). Random seeds are fixed.